# Chapter 21: EKF-SLAM

<a href="../lite/lab/index.html?path=ch21_ekf_slam.ipynb" target="_blank" style="display:inline-block;padding:8px 16px;background:#1976d2;color:white;border-radius:4px;text-decoration:none;font-weight:bold">▶ Open in JupyterLite (editable, no install)</a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

def draw_cov_ellipse(ax, mean, cov, n_std=2, **kwargs):
    from matplotlib.patches import Ellipse
    vals, vecs = np.linalg.eigh(cov)
    angle = np.degrees(np.arctan2(vecs[1,1], vecs[0,1]))
    w, h = 2 * n_std * np.sqrt(np.maximum(vals, 0))
    ax.add_patch(Ellipse(xy=mean, width=w, height=h, angle=angle, **kwargs))

Take the Kalman filter from Chapter 16. Now, instead of tracking just the robot,
stuff every landmark into the state vector too. The state grows. The covariance
matrix grows quadratically. But something magical happens: every time you see a
landmark, the uncertainty of EVERY landmark in the map decreases. The entire map
learns from every single observation.

This chapter implements **EKF-SLAM** from scratch: the prediction step, the
update step, landmark initialization, and a full simulation showing the map
converging as the robot drives a loop.

```{admonition} What you will build
:class: tip

- Implement full EKF-SLAM for a 2D robot observing 10 point landmarks
- Watch the map converge as the robot drives a loop and observes landmarks
- See the dramatic improvement when the robot closes the loop
- Understand the O(N^2) scalability limit that constrains EKF-SLAM to small maps

**Real world application:** EKF-SLAM was the first practical SLAM algorithm and is still used in small scale applications. After this chapter, you will have built a complete SLAM system from scratch.
```

```{admonition} Libraries and tools used in practice
:class: note

In this chapter we implement everything from scratch for learning. In production, engineers use these libraries:

| Library / Tool | What it does |
|---|---|
| **mrpt::slam::CRangeBearingKFSLAM2D** | C++ EKF-SLAM implementation |
| **GTSAM** | Can implement EKF-SLAM as a special case of factor graph optimization |

Implementing from scratch teaches you **why** these tools work. Using them in production saves you from reinventing the wheel.
```

## 21.1 Joint State: Robot + Landmarks

In EKF-SLAM, the state vector holds the robot pose **and** every landmark position:

$$\mathbf{x} = \begin{bmatrix} x_R \\ y_R \\ \theta_R \\ l_{1x} \\ l_{1y} \\ \vdots \\ l_{Nx} \\ l_{Ny} \end{bmatrix}, \qquad
P = \begin{bmatrix} P_{RR} & P_{RL_1} & \cdots & P_{RL_N} \\
P_{L_1R} & P_{L_1L_1} & \cdots & P_{L_1L_N} \\
\vdots & \vdots & \ddots & \vdots \\
P_{L_NR} & P_{L_NL_1} & \cdots & P_{L_NL_N} \end{bmatrix}$$

The state dimension is $3 + 2N$ for $N$ landmarks. The covariance is
$(3+2N) \times (3+2N)$.

In [ ]:
# ── PARAMETERS ── change these and re-run ─────────────────────────────────────
n_landmarks = 4
robot_init = np.array([1.0, 1.0, 0.0])       # x, y, theta
landmark_true = np.array([[3, 7], [7, 8], [9, 3], [5, 1]], dtype=float)
# ──────────────────────────────────────────────────────────────────────────────

state_dim = 3 + 2 * n_landmarks

# Initial state: robot known, landmarks initialized with high uncertainty
x = np.zeros(state_dim)
x[:3] = robot_init
for i in range(n_landmarks):
    x[3 + 2*i]     = landmark_true[i, 0] + np.random.randn() * 1.0
    x[3 + 2*i + 1] = landmark_true[i, 1] + np.random.randn() * 1.0

P = np.eye(state_dim)
P[0,0] = P[1,1] = 0.01  # robot position well known
P[2,2] = 0.01           # heading well known
for i in range(n_landmarks):
    P[3+2*i, 3+2*i] = 5.0
    P[3+2*i+1, 3+2*i+1] = 5.0

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ax = axes[0]
ax.plot(x[0], x[1], 'ko', ms=10, zorder=5, label='Robot')
draw_cov_ellipse(ax, x[:2], P[:2,:2], fill=True, facecolor='steelblue',
                 alpha=0.3, edgecolor='steelblue', lw=2)
for i in range(n_landmarks):
    est = x[3+2*i:3+2*i+2]
    cov_lm = P[3+2*i:3+2*i+2, 3+2*i:3+2*i+2]
    ax.scatter(*landmark_true[i], c='forestgreen', s=100, marker='*', zorder=5)
    ax.scatter(*est, c='tomato', s=60, marker='^', zorder=5)
    draw_cov_ellipse(ax, est, cov_lm, fill=True, facecolor='tomato',
                     alpha=0.15, edgecolor='tomato', lw=1.5)
    ax.annotate(f'L{i}', xy=est, xytext=(5,5), textcoords='offset points',
                fontsize=10, color='tomato')
ax.set_xlim(-1, 12); ax.set_ylim(-1, 12); ax.set_aspect('equal')
ax.set_title('Initial SLAM state (large landmark uncertainty)', fontsize=13)
ax.legend(loc='upper left')

ax = axes[1]
im = ax.imshow(np.abs(P), cmap='Blues', interpolation='nearest')
labels = ['Rx','Ry','Rθ'] + [f'L{i}{c}' for i in range(n_landmarks) for c in ['x','y']]
ax.set_xticks(range(state_dim)); ax.set_xticklabels(labels, fontsize=7, rotation=45)
ax.set_yticks(range(state_dim)); ax.set_yticklabels(labels, fontsize=7)
ax.set_title('Initial covariance (diagonal, no correlations)', fontsize=13)
plt.colorbar(im, ax=ax, shrink=0.8)

plt.tight_layout()
plt.show()

print(f'State dimension: {state_dim}')
print(f'Covariance size: {state_dim}×{state_dim} = {state_dim**2} entries')

**Observation:** The covariance starts diagonal because we have no information
linking the robot to the landmarks yet. That changes as soon as the robot begins
observing them.

## 21.2 Cross-Covariance: Why Landmarks Become Correlated

When the robot observes landmark $L_j$, the Kalman update modifies the entire
covariance matrix. The key insight: the **cross-covariance** blocks $P_{RL_j}$
and $P_{L_iL_j}$ become nonzero. Through the robot, every landmark is linked
to every other landmark.

The observation model for a range-bearing sensor is:

$$\mathbf{z}_j = \begin{bmatrix} \sqrt{(l_{jx} - x_R)^2 + (l_{jy} - y_R)^2} \\
\mathrm{atan2}(l_{jy} - y_R,\; l_{jx} - x_R) - \theta_R \end{bmatrix} + \mathbf{v}_j$$

In [ ]:
def observation_model(robot, landmark):
    """Compute range-bearing observation from robot to landmark."""
    dx = landmark[0] - robot[0]
    dy = landmark[1] - robot[1]
    r = np.sqrt(dx**2 + dy**2)
    phi = np.arctan2(dy, dx) - robot[2]
    phi = (phi + np.pi) % (2*np.pi) - np.pi   # wrap to [-pi, pi]
    return np.array([r, phi])

def observation_jacobian(robot, landmark, lm_idx, state_dim):
    """Jacobian of observation model w.r.t. full state."""
    dx = landmark[0] - robot[0]
    dy = landmark[1] - robot[1]
    q = dx**2 + dy**2
    r = np.sqrt(q)
    
    H = np.zeros((2, state_dim))
    # w.r.t. robot [x, y, theta]
    H[0, 0] = -dx / r;  H[0, 1] = -dy / r;  H[0, 2] = 0
    H[1, 0] =  dy / q;  H[1, 1] = -dx / q;  H[1, 2] = -1
    # w.r.t. landmark [lx, ly]
    li = 3 + 2 * lm_idx
    H[0, li]   =  dx / r;  H[0, li+1] =  dy / r
    H[1, li]   = -dy / q;  H[1, li+1] =  dx / q
    return H

print('Observation and Jacobian functions defined.')
print('Test: observe L0 from robot at (1,1,0):')
z_test = observation_model(robot_init, landmark_true[0])
print(f'  range = {z_test[0]:.2f} m, bearing = {np.degrees(z_test[1]):.1f} deg')

In [ ]:
# ── PARAMETERS ── change these and re-run ─────────────────────────────────────
n_updates = 20        # total observations to simulate (try 1, 5, 20)
sigma_r = 0.3         # range noise std
sigma_phi = 0.1       # bearing noise std
# ──────────────────────────────────────────────────────────────────────────────

np.random.seed(42)
R_obs = np.diag([sigma_r**2, sigma_phi**2])

# Reset state and covariance
x_sim = np.zeros(state_dim)
x_sim[:3] = robot_init.copy()
for i in range(n_landmarks):
    x_sim[3+2*i:3+2*i+2] = landmark_true[i] + np.random.randn(2)*1.5

P_sim = np.eye(state_dim) * 0.01
for i in range(n_landmarks):
    P_sim[3+2*i, 3+2*i] = 5.0
    P_sim[3+2*i+1, 3+2*i+1] = 5.0

# Run EKF updates
for k in range(n_updates):
    lm_idx = k % n_landmarks
    z_true = observation_model(robot_init, landmark_true[lm_idx])
    z_noisy = z_true + np.array([np.random.randn()*sigma_r, np.random.randn()*sigma_phi])
    
    lm_est = x_sim[3+2*lm_idx:3+2*lm_idx+2]
    z_pred = observation_model(x_sim[:3], lm_est)
    
    H = observation_jacobian(x_sim[:3], lm_est, lm_idx, state_dim)
    S = H @ P_sim @ H.T + R_obs
    K = P_sim @ H.T @ np.linalg.inv(S)
    
    innovation = z_noisy - z_pred
    innovation[1] = (innovation[1] + np.pi) % (2*np.pi) - np.pi
    
    x_sim = x_sim + K @ innovation
    P_sim = (np.eye(state_dim) - K @ H) @ P_sim

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ax = axes[0]
ax.plot(x_sim[0], x_sim[1], 'ko', ms=10, zorder=5)
draw_cov_ellipse(ax, x_sim[:2], P_sim[:2,:2], fill=True,
                 facecolor='steelblue', alpha=0.3, edgecolor='steelblue', lw=2)
for i in range(n_landmarks):
    est = x_sim[3+2*i:3+2*i+2]
    cov_lm = P_sim[3+2*i:3+2*i+2, 3+2*i:3+2*i+2]
    ax.scatter(*landmark_true[i], c='forestgreen', s=120, marker='*', zorder=5)
    ax.scatter(*est, c='tomato', s=60, marker='^', zorder=5)
    draw_cov_ellipse(ax, est, cov_lm, fill=True,
                     facecolor='tomato', alpha=0.15, edgecolor='tomato', lw=1.5)
ax.set_xlim(-1, 12); ax.set_ylim(-1, 12); ax.set_aspect('equal')
ax.set_title(f'After {n_updates} updates: ellipses shrink!', fontsize=13)

ax = axes[1]
im = ax.imshow(np.abs(P_sim), cmap='Blues', interpolation='nearest')
labels = ['Rx','Ry','Rθ'] + [f'L{i}{c}' for i in range(n_landmarks) for c in ['x','y']]
ax.set_xticks(range(state_dim)); ax.set_xticklabels(labels, fontsize=7, rotation=45)
ax.set_yticks(range(state_dim)); ax.set_yticklabels(labels, fontsize=7)
ax.set_title(f'Covariance after {n_updates} updates (off-diagonal = correlations)', fontsize=13)
plt.colorbar(im, ax=ax, shrink=0.8)

plt.tight_layout()
plt.show()

**Key insight:** The off-diagonal blocks of the covariance are now nonzero.
Landmarks that have never been observed **together** still become correlated,
because the robot acts as a conduit. Observing $L_0$ improves the robot estimate,
which in turn improves the estimate of $L_1$ through their shared cross-covariance.

## 21.3 EKF-SLAM Prediction and Update

The full EKF-SLAM algorithm has two steps per timestep.

**Prediction** (robot moves):
$$\bar{\mathbf{x}}_R = f(\mathbf{x}_R, \mathbf{u}), \qquad
\bar{P} = F \, P \, F^T + \bar{Q}$$

where $F$ is the Jacobian of the motion model (identity for landmarks) and
$\bar{Q}$ adds noise only to the robot block.

**Update** (robot observes landmark $j$):
$$K = \bar{P} \, H_j^T (H_j \, \bar{P} \, H_j^T + R)^{-1}$$
$$\mathbf{x} = \bar{\mathbf{x}} + K (\mathbf{z}_j - h_j(\bar{\mathbf{x}}))$$
$$P = (I - K H_j) \bar{P}$$

In [ ]:
def motion_model(state, u, dt=1.0):
    """Velocity motion model: u = [v, omega]."""
    x, y, theta = state[0], state[1], state[2]
    v, omega = u
    if abs(omega) < 1e-6:
        x_new = x + v * dt * np.cos(theta)
        y_new = y + v * dt * np.sin(theta)
        theta_new = theta
    else:
        x_new = x + v/omega * (np.sin(theta + omega*dt) - np.sin(theta))
        y_new = y + v/omega * (np.cos(theta) - np.cos(theta + omega*dt))
        theta_new = theta + omega * dt
    theta_new = (theta_new + np.pi) % (2*np.pi) - np.pi
    new_state = state.copy()
    new_state[0] = x_new
    new_state[1] = y_new
    new_state[2] = theta_new
    return new_state

def motion_jacobian(state, u, n_state, dt=1.0):
    """Jacobian of motion model w.r.t. full state."""
    theta = state[2]
    v, omega = u
    F = np.eye(n_state)
    if abs(omega) < 1e-6:
        F[0, 2] = -v * dt * np.sin(theta)
        F[1, 2] =  v * dt * np.cos(theta)
    else:
        F[0, 2] = v/omega * (np.cos(theta + omega*dt) - np.cos(theta))
        F[1, 2] = v/omega * (np.sin(theta + omega*dt) - np.sin(theta))
    return F

print('Motion model and Jacobian defined.')
print('Test: move from (1,1,0) with v=1, omega=0.3:')
test_state = np.zeros(state_dim); test_state[:3] = [1, 1, 0]
moved = motion_model(test_state, [1.0, 0.3])
print(f'  New pose: ({moved[0]:.2f}, {moved[1]:.2f}, {np.degrees(moved[2]):.1f} deg)')

In [ ]:
def ekf_slam_predict(x, P, u, Q, dt=1.0):
    """EKF-SLAM prediction step."""
    n = len(x)
    x_pred = motion_model(x, u, dt)
    F = motion_jacobian(x, u, n, dt)
    # Add process noise only to robot block
    Q_full = np.zeros((n, n))
    Q_full[:3, :3] = Q
    P_pred = F @ P @ F.T + Q_full
    return x_pred, P_pred

def ekf_slam_update(x, P, z, lm_idx, R):
    """EKF-SLAM update step for one landmark observation."""
    n = len(x)
    lm_est = x[3+2*lm_idx:3+2*lm_idx+2]
    z_pred = observation_model(x[:3], lm_est)
    H = observation_jacobian(x[:3], lm_est, lm_idx, n)
    
    innovation = z - z_pred
    innovation[1] = (innovation[1] + np.pi) % (2*np.pi) - np.pi
    
    S = H @ P @ H.T + R
    K = P @ H.T @ np.linalg.inv(S)
    x_new = x + K @ innovation
    x_new[2] = (x_new[2] + np.pi) % (2*np.pi) - np.pi
    P_new = (np.eye(n) - K @ H) @ P
    return x_new, P_new

print('EKF-SLAM predict and update functions ready.')

**Note:** The motion noise $Q$ only enters the top-left $3 \times 3$ block.
Landmarks do not move, so their process noise is zero. But during prediction,
the cross-covariance blocks $P_{RL_i}$ grow because the motion Jacobian $F$
has off-diagonal structure through the robot heading dependence.

## 21.4 Scalability Limits: $O(N^2)$ Growth

The covariance matrix has size $(3 + 2N) \times (3 + 2N)$. Each update requires
computing $K = P H^T S^{-1}$, which is $O(N^2)$ because $P$ is full.

For 100 landmarks: $203 \times 203 = 41{,}209$ entries.
For 1000 landmarks: $2003 \times 2003 = 4{,}012{,}009$ entries.
For 10,000 landmarks: $20{,}003 \times 20{,}003 \approx 400$ million entries.

This quadratic growth is the fundamental limitation of EKF-SLAM.

In [ ]:
# ── PARAMETERS ── change these and re-run ─────────────────────────────────────
landmark_counts = [10, 50, 100, 200, 500, 1000]
# ──────────────────────────────────────────────────────────────────────────────

state_dims = [3 + 2*n for n in landmark_counts]
cov_entries = [d**2 for d in state_dims]
update_ops = [d**2 for d in state_dims]  # dominant cost

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(landmark_counts, [d for d in state_dims], 'steelblue', lw=2, marker='o')
ax.set_xlabel('Number of landmarks', fontsize=12)
ax.set_ylabel('State dimension', fontsize=12)
ax.set_title('State dimension grows linearly: $3 + 2N$', fontsize=13)

ax = axes[1]
ax.semilogy(landmark_counts, cov_entries, 'tomato', lw=2, marker='s')
ax.set_xlabel('Number of landmarks', fontsize=12)
ax.set_ylabel('Covariance entries', fontsize=12)
ax.set_title('Covariance size grows quadratically: $(3+2N)^2$', fontsize=13)

plt.tight_layout()
plt.show()

for n, d, c in zip(landmark_counts, state_dims, cov_entries):
    print(f'  {n:5d} landmarks → state dim {d:5d}, cov entries {c:>12,}')

In [ ]:
# ── PARAMETERS ── change these and re-run ─────────────────────────────────────
test_sizes = [10, 25, 50, 100, 200]
# ──────────────────────────────────────────────────────────────────────────────

import time

times_predict = []
times_update = []

for n_lm in test_sizes:
    sd = 3 + 2 * n_lm
    x_t = np.random.randn(sd)
    P_t = np.eye(sd) * 2.0
    Q_t = np.diag([0.1, 0.1, 0.01])
    R_t = np.diag([0.1, 0.05])
    u_t = [1.0, 0.1]
    
    t0 = time.perf_counter()
    for _ in range(10):
        x_p, P_p = ekf_slam_predict(x_t, P_t, u_t, Q_t)
    times_predict.append((time.perf_counter() - t0) / 10)
    
    t0 = time.perf_counter()
    for _ in range(10):
        x_u, P_u = ekf_slam_update(x_t, P_t, np.array([1.0, 0.1]), 0, R_t)
    times_update.append((time.perf_counter() - t0) / 10)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(test_sizes, [t*1000 for t in times_predict], 'steelblue', lw=2, marker='o', label='Predict')
ax.plot(test_sizes, [t*1000 for t in times_update], 'tomato', lw=2, marker='s', label='Update')
ax.set_xlabel('Number of landmarks', fontsize=12)
ax.set_ylabel('Time per step (ms)', fontsize=12)
ax.set_title('EKF-SLAM computation time vs map size', fontsize=13)
ax.legend()
plt.tight_layout()
plt.show()

**Observation:** Both prediction and update scale roughly as $O(N^2)$.
At 200 landmarks the cost is already significant. At 10,000 landmarks,
EKF-SLAM becomes impractical for real time operation. This motivates
the particle based approach in Chapter 22 and the graph based approach
in Chapters 24 and 25.

---

## Capstone: Full EKF-SLAM Simulation

A 2D robot drives a loop past 10 landmarks. We run the complete EKF-SLAM
algorithm (predict + update) at every timestep. Watch the map converge:
ellipses shrink, and at **loop closure** (returning to the start), the
improvement is dramatic.

In [ ]:
# ── PARAMETERS ── change these and re-run ─────────────────────────────────────
np.random.seed(7)
N_LM = 10
sigma_v = 0.15       # velocity noise
sigma_om = 0.05      # angular velocity noise
sigma_range = 0.5    # range measurement noise
sigma_bear = 0.15    # bearing measurement noise
max_obs_range = 6.0  # maximum observation range
dt = 1.0
# ──────────────────────────────────────────────────────────────────────────────

# Place landmarks in a ring
lm_angles = np.linspace(0, 2*np.pi, N_LM, endpoint=False)
lm_radius = 7.0
landmarks_gt = np.column_stack([lm_radius*np.cos(lm_angles),
                                 lm_radius*np.sin(lm_angles)])

# Robot drives a smaller circle (inside the ring of landmarks)
n_steps = 40
drive_radius = 4.0
drive_omega = 2 * np.pi / n_steps
drive_v = drive_radius * drive_omega

# Initialize state: robot only (landmarks added when first observed)
true_robot = np.array([drive_radius, 0.0, np.pi/2])
x_ekf = np.array([drive_radius, 0.0, np.pi/2])
P_ekf = np.diag([0.01, 0.01, 0.01])

Q_motion = np.diag([sigma_v**2, sigma_v**2, sigma_om**2])
R_meas = np.diag([sigma_range**2, sigma_bear**2])

landmark_seen = [False] * N_LM
lm_state_idx = {}  # maps landmark index to position in state vector

true_path = [true_robot[:2].copy()]
est_path = [x_ekf[:2].copy()]
snapshots = {}  # store state at specific times

for t in range(n_steps):
    u = np.array([drive_v, drive_omega])
    
    # True motion
    true_next = np.zeros(3)
    true_next[0] = true_robot[0] + drive_v/drive_omega * (
        np.sin(true_robot[2] + drive_omega*dt) - np.sin(true_robot[2]))
    true_next[1] = true_robot[1] + drive_v/drive_omega * (
        np.cos(true_robot[2]) - np.cos(true_robot[2] + drive_omega*dt))
    true_next[2] = true_robot[2] + drive_omega * dt
    true_robot = true_next
    true_path.append(true_robot[:2].copy())
    
    # EKF Predict
    n_s = len(x_ekf)
    x_ekf = motion_model(x_ekf, u, dt)
    F = motion_jacobian(x_ekf, u, n_s, dt)
    Q_full = np.zeros((n_s, n_s))
    Q_full[:3, :3] = Q_motion
    P_ekf = F @ P_ekf @ F.T + Q_full
    
    # Observe landmarks in range
    for j in range(N_LM):
        dx = landmarks_gt[j, 0] - true_robot[0]
        dy = landmarks_gt[j, 1] - true_robot[1]
        dist = np.sqrt(dx**2 + dy**2)
        if dist > max_obs_range:
            continue
        
        # Generate noisy measurement
        z_true = observation_model(true_robot, landmarks_gt[j])
        z_meas = z_true + np.array([np.random.randn()*sigma_range,
                                     np.random.randn()*sigma_bear])
        
        if not landmark_seen[j]:
            # Initialize new landmark in state
            lm_state_idx[j] = (len(x_ekf) - 3) // 2
            # Estimate landmark position from measurement
            lm_x = x_ekf[0] + z_meas[0] * np.cos(z_meas[1] + x_ekf[2])
            lm_y = x_ekf[1] + z_meas[0] * np.sin(z_meas[1] + x_ekf[2])
            x_ekf = np.append(x_ekf, [lm_x, lm_y])
            # Expand covariance
            n_new = len(x_ekf)
            P_new = np.eye(n_new) * 10.0
            P_new[:n_new-2, :n_new-2] = P_ekf
            P_ekf = P_new
            landmark_seen[j] = True
        
        # EKF Update
        li = lm_state_idx[j]
        n_s = len(x_ekf)
        lm_est = x_ekf[3+2*li:3+2*li+2]
        z_pred = observation_model(x_ekf[:3], lm_est)
        H = observation_jacobian(x_ekf[:3], lm_est, li, n_s)
        
        innov = z_meas - z_pred
        innov[1] = (innov[1] + np.pi) % (2*np.pi) - np.pi
        
        S = H @ P_ekf @ H.T + R_meas
        K = P_ekf @ H.T @ np.linalg.inv(S)
        x_ekf = x_ekf + K @ innov
        x_ekf[2] = (x_ekf[2] + np.pi) % (2*np.pi) - np.pi
        P_ekf = (np.eye(n_s) - K @ H) @ P_ekf
    
    est_path.append(x_ekf[:2].copy())
    
    # Store snapshots
    if t in [5, 20, n_steps-1]:
        snapshots[t] = (x_ekf.copy(), P_ekf.copy())

true_path = np.array(true_path)
est_path = np.array(est_path)

print(f'Final state dimension: {len(x_ekf)}')
print(f'Landmarks observed: {sum(landmark_seen)} / {N_LM}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

snap_labels = {5: 'Early (t=5)', 20: 'Mid (t=20)', n_steps-1: f'Final (t={n_steps-1}, loop closed)'}

for ax, (t_snap, (x_s, P_s)) in zip(axes, snapshots.items()):
    # True landmarks
    ax.scatter(landmarks_gt[:, 0], landmarks_gt[:, 1], c='forestgreen',
              s=100, marker='*', zorder=5, label='True landmarks')
    # Estimated landmarks
    n_lm_seen = (len(x_s) - 3) // 2
    for i in range(n_lm_seen):
        est = x_s[3+2*i:3+2*i+2]
        cov_lm = P_s[3+2*i:3+2*i+2, 3+2*i:3+2*i+2]
        ax.scatter(*est, c='tomato', s=40, marker='^', zorder=5)
        draw_cov_ellipse(ax, est, cov_lm, n_std=2, fill=True,
                         facecolor='tomato', alpha=0.15, edgecolor='tomato', lw=1)
    
    # Paths
    ax.plot(true_path[:t_snap+2, 0], true_path[:t_snap+2, 1],
            'k--', lw=1, alpha=0.5, label='True path')
    ax.plot(est_path[:t_snap+2, 0], est_path[:t_snap+2, 1],
            'steelblue', lw=2, label='Estimated path')
    
    ax.set_xlim(-10, 10); ax.set_ylim(-10, 10); ax.set_aspect('equal')
    ax.set_title(snap_labels[t_snap], fontsize=13)
    ax.legend(fontsize=8, loc='upper left')

plt.suptitle('EKF-SLAM: Map convergence over time', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Visualize covariance matrices at the three snapshots
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (t_snap, (x_s, P_s)) in zip(axes, snapshots.items()):
    im = ax.imshow(np.log10(np.abs(P_s) + 1e-10), cmap='Blues',
                   interpolation='nearest', vmin=-4, vmax=1)
    ax.set_title(f't = {t_snap}: {P_s.shape[0]}×{P_s.shape[0]}', fontsize=13)
    plt.colorbar(im, ax=ax, shrink=0.7, label='log10(|P|)')

plt.suptitle('Covariance matrix at 3 timestamps (log scale)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print('Notice: the off-diagonal blocks grow denser over time.')
print('At loop closure, the entire matrix tightens up.')

In [ ]:
# Final landmark errors
print('Landmark estimation errors at final timestep:')
print(f'{"LM":>4} {"True":>16} {"Estimated":>16} {"Error (m)":>10}')
print('-' * 50)
n_lm_final = (len(x_ekf) - 3) // 2
errors = []
for j_seen, j_true in enumerate(sorted(lm_state_idx.keys())):
    li = lm_state_idx[j_true]
    est = x_ekf[3+2*li:3+2*li+2]
    true = landmarks_gt[j_true]
    err = np.linalg.norm(est - true)
    errors.append(err)
    print(f'  L{j_true}: ({true[0]:6.2f},{true[1]:6.2f})  '
          f'({est[0]:6.2f},{est[1]:6.2f})  {err:8.3f}')
print(f'\nMean error: {np.mean(errors):.3f} m')
print(f'Max error:  {np.max(errors):.3f} m')

**Capstone observations:**
- Early in the trajectory, only nearby landmarks have been observed and their ellipses are large.
- As the robot loops around, more landmarks are incorporated and all ellipses shrink.
- At **loop closure**, revisiting landmarks from the start produces a dramatic correction: the cross-covariances propagate the improvement to every landmark in the map.
- The covariance matrix at the final timestep is densely filled with correlations.
- For 10 landmarks, EKF-SLAM runs quickly. For thousands of landmarks, a different approach is needed (see Chapter 22).

---

## Exercises

### Exercise 21.1: Vary the number of landmarks

Run the capstone simulation with 5, 10, and 20 landmarks. How does the final
mean landmark error change? Does more landmarks improve or hurt accuracy?

In [ ]:
# Your code here
# Modify N_LM and rerun the capstone loop

### Exercise 21.2: Effect of sensor noise

Keep 10 landmarks. Try sigma_range = 0.1, 0.5, and 2.0.
Plot the final landmark error for each setting. How sensitive is EKF-SLAM to sensor quality?

In [ ]:
# Your code here

### Exercise 21.3: What happens without loop closure? (challenge)

Modify the robot trajectory so it drives a straight line instead of a loop.
Compare the final covariance matrix structure with the loop trajectory.
Which landmarks have the smallest uncertainty? Why?

In [ ]:
# Your code here